In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# Project Variables

lakehouse_name = "lh_dev_ecommerce"
env_name = "dev"
job_id = 105

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 3, Finished, Available, Finished, False)

In [2]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {lakehouse_name}.{env_name}_gold_db
""")

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 4, Finished, Available, Finished, False)

DataFrame[]

In [3]:
customers_df = spark.table(
    f"{lakehouse_name}.{env_name}_silver_db.customers"
)

orders_df = spark.table(
    f"{lakehouse_name}.{env_name}_silver_db.orders"
)

products_df = spark.table(
    f"{lakehouse_name}.{env_name}_silver_db.products"
)

order_items_df = spark.table(
    f"{lakehouse_name}.{env_name}_silver_db.order_items"
)

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 5, Finished, Available, Finished, False)

In [4]:
print("Customers :", customers_df.count())
print("Orders :", orders_df.count())
print("Products :", products_df.count())
print("Order Items :", order_items_df.count())

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 6, Finished, Available, Finished, False)

Customers : 26
Orders : 25
Products : 25
Order Items : 37


In [5]:
from pyspark.sql.functions import sum, round

gold_sales_region = (
    orders_df.groupBy("Region")
    .agg(
        round(sum("Price"), 2).alias("TotalSales"),
    )
    .orderBy("TotalSales", ascending=False)
)

gold_sales_region.show(truncate=False)

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 7, Finished, Available, Finished, False)

+------+----------+
|Region|TotalSales|
+------+----------+
|North |6396.26   |
|West  |3906.72   |
|East  |3341.98   |
|South |1161.12   |
+------+----------+



In [6]:
gold_sales_region.write \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable(
    f"{lakehouse_name}.{env_name}_gold_db.gold_sales_by_region"
)

print("Gold Table Created Successfully")

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 8, Finished, Available, Finished, False)

Gold Table Created Successfully


In [7]:
spark.sql(f"""
SELECT *
FROM {lakehouse_name}.{env_name}_gold_db.gold_sales_by_region
ORDER BY TotalSales DESC
""").show(truncate=False)

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 9, Finished, Available, Finished, False)

+------+----------+
|Region|TotalSales|
+------+----------+
|North |6396.26   |
|West  |3906.72   |
|East  |3341.98   |
|South |1161.12   |
+------+----------+



In [8]:
from pyspark.sql.functions import sum, round

gold_top_customers = (
    orders_df.join(
        customers_df,
        on="CustomerID",
        how="inner"
    )
    .groupBy(
        "CustomerID",
        "CustomerName"
    )
    .agg(
        round(sum("Price"), 2).alias("TotalSales")
    )
    .orderBy("TotalSales", ascending=False)
)

gold_top_customers.show(truncate=False)

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 10, Finished, Available, Finished, False)

+----------+------------+----------+
|CustomerID|CustomerName|TotalSales|
+----------+------------+----------+
|CUST018   |Sneha Kapoor|1688.51   |
|CUST009   |Arjun Mehta |1523.74   |
|CUST010   |Priya Nair  |1379.19   |
|CUST020   |Rohit Sharma|1331.65   |
|CUST002   |Rohit Sharma|986.78    |
|CUST005   |Arjun Mehta |971.84    |
|CUST022   |Arjun Mehta |895.72    |
|CUST012   |Ananya Iyer |877.45    |
|CUST004   |Rohit Sharma|822.56    |
|CUST014   |Ananya Iyer |804.15    |
|CUST006   |Rahul Verma |802.38    |
|CUST019   |Priya Nair  |737.91    |
|CUST013   |Arjun Mehta |621.03    |
|CUST016   |Rahul Verma |565.88    |
|CUST024   |Ananya Iyer |318.7     |
|CUST011   |Ananya Iyer |314.82    |
|CUST008   |Priya Nair  |163.77    |
+----------+------------+----------+



In [9]:
gold_top_customers.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable(
    f"{lakehouse_name}.{env_name}_gold_db.gold_top_customers"
)

print("gold_top_customers created successfully")

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 11, Finished, Available, Finished, False)

gold_top_customers created successfully


In [10]:
spark.sql(f"""
SELECT *
FROM {lakehouse_name}.{env_name}_gold_db.gold_top_customers
ORDER BY TotalSales DESC
""").show(truncate=False)

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 12, Finished, Available, Finished, False)

+----------+------------+----------+
|CustomerID|CustomerName|TotalSales|
+----------+------------+----------+
|CUST018   |Sneha Kapoor|1688.51   |
|CUST009   |Arjun Mehta |1523.74   |
|CUST010   |Priya Nair  |1379.19   |
|CUST020   |Rohit Sharma|1331.65   |
|CUST002   |Rohit Sharma|986.78    |
|CUST005   |Arjun Mehta |971.84    |
|CUST022   |Arjun Mehta |895.72    |
|CUST012   |Ananya Iyer |877.45    |
|CUST004   |Rohit Sharma|822.56    |
|CUST014   |Ananya Iyer |804.15    |
|CUST006   |Rahul Verma |802.38    |
|CUST019   |Priya Nair  |737.91    |
|CUST013   |Arjun Mehta |621.03    |
|CUST016   |Rahul Verma |565.88    |
|CUST024   |Ananya Iyer |318.7     |
|CUST011   |Ananya Iyer |314.82    |
|CUST008   |Priya Nair  |163.77    |
+----------+------------+----------+



In [11]:
from pyspark.sql.functions import year, month, sum, col

monthly_sales = spark.table(
    f"{lakehouse_name}.{env_name}_silver_db.orders"
).groupBy(
    year(col("OrderDate")).alias("Year"),
    month(col("OrderDate")).alias("Month")
).agg(
    sum(col("Price").cast("double")).alias("TotalSales")
)

monthly_sales.show(truncate=False)

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 13, Finished, Available, Finished, False)

+----+-----+----------+
|Year|Month|TotalSales|
+----+-----+----------+
|2023|8    |1482.27   |
|2023|9    |151.19    |
|2023|6    |747.78    |
|2023|3    |3177.09   |
|2023|2    |2829.14   |
|2023|4    |2770.44   |
|2023|5    |897.48    |
|2023|10   |1100.13   |
|2023|1    |1650.56   |
+----+-----+----------+



In [14]:
gold_monthly_sales = spark.sql(f"""
SELECT
    YEAR(OrderDate) AS Year,
    MONTH(OrderDate) AS Month,
    SUM(CAST(Price AS DOUBLE)) AS TotalSales
FROM {lakehouse_name}.{env_name}_silver_db.orders
GROUP BY
    YEAR(OrderDate),
    MONTH(OrderDate)
ORDER BY
    Year, Month
""")

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 16, Finished, Available, Finished, False)

In [16]:
gold_monthly_sales.write \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable(
    f"{lakehouse_name}.{env_name}_gold_db.gold_monthly_sales"
)

print("gold_monthly_sales created successfully")

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 18, Finished, Available, Finished, False)

gold_monthly_sales created successfully


In [17]:
spark.sql(f"""
SELECT
    OrderID,
    OrderDate
FROM {lakehouse_name}.{env_name}_silver_db.orders
LIMIT 10
""").show(truncate=False)

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 19, Finished, Available, Finished, False)

+-------+----------+
|OrderID|OrderDate |
+-------+----------+
|ORD011 |2023-10-16|
|ORD021 |2023-10-25|
|ORD023 |2023-06-10|
|ORD010 |2023-05-14|
|ORD006 |2023-03-26|
|ORD007 |2023-04-08|
|ORD025 |2023-04-18|
|ORD004 |2023-05-08|
|ORD012 |2023-03-22|
|ORD009 |2023-03-03|
+-------+----------+



In [18]:
product_sales = spark.sql(f"""
SELECT
    oi.ProductID,
    p.Category,
    p.Carrier,
    SUM(CAST(o.Price AS DOUBLE) * CAST(oi.Quantity AS INT)) AS TotalRevenue,
    SUM(CAST(oi.Quantity AS INT)) AS TotalQuantitySold
FROM {lakehouse_name}.{env_name}_silver_db.order_items oi
JOIN {lakehouse_name}.{env_name}_silver_db.orders o
    ON oi.OrderID = o.OrderID
JOIN {lakehouse_name}.{env_name}_silver_db.products p
    ON oi.ProductID = p.ProductID
GROUP BY
    oi.ProductID,
    p.Category,
    p.Carrier
ORDER BY TotalRevenue DESC
""")

product_sales.show(truncate=False)

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 20, Finished, Available, Finished, False)

+---------+-----------+------------+------------------+-----------------+
|ProductID|Category   |Carrier     |TotalRevenue      |TotalQuantitySold|
+---------+-----------+------------+------------------+-----------------+
|PROD022  |Books      |Delhivery   |11135.68          |18               |
|PROD013  |Books      |Ecom Express|8172.77           |12               |
|PROD023  |Home       |Delhivery   |7177.24           |8                |
|PROD002  |Clothing   |Delhivery   |4859.2            |5                |
|PROD007  |Books      |Delhivery   |4129.38           |7                |
|PROD004  |Books      |Delhivery   |4097.03           |11               |
|PROD017  |Electronics|Ecom Express|3543.6            |8                |
|PROD016  |Clothing   |Blue Dart   |3409.68           |4                |
|PROD001  |Electronics|Blue Dart   |3354.6000000000004|7                |
|PROD011  |Electronics|Delhivery   |3305.94           |5                |
|PROD019  |Electronics|DTDC        |31

In [19]:
product_sales.write \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable(
    f"{lakehouse_name}.{env_name}_gold_db.gold_product_performance"
)

print("Product Performance Gold Table Created Successfully")

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 21, Finished, Available, Finished, False)

Product Performance Gold Table Created Successfully


In [20]:
spark.sql(f"""
SELECT *
FROM {lakehouse_name}.{env_name}_silver_db.customers
LIMIT 5
""").show(truncate=False)

spark.sql(f"""
SELECT *
FROM {lakehouse_name}.{env_name}_silver_db.orders
LIMIT 5
""").show(truncate=False)

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 22, Finished, Available, Finished, False)

+----------+------------+-----------------+----------+--------------------------+
|CustomerID|CustomerName|IsPremiumCustomer|SignupDate|InsertTimestamp           |
+----------+------------+-----------------+----------+--------------------------+
|CUST010   |Priya Nair  |NO               |2022-12-15|2026-07-09 08:57:47.161559|
|CUST015   |Rohit Sharma|NO               |2023-07-04|2026-07-09 08:57:47.161559|
|CUST002   |Rohit Sharma|NO               |2022-09-08|2026-07-09 08:57:47.161559|
|CUST011   |Ananya Iyer |YES              |2022-08-09|2026-07-09 08:57:47.161559|
|CUST009   |Arjun Mehta |NO               |2022-01-07|2026-07-09 08:57:47.161559|
+----------+------------+-----------------+----------+--------------------------+

+-------+----------+----------+-----------+-------------+------------------+------+------------+----------+-------------------------+
|OrderID|CustomerID|OrderDate |ShippedDate|DeliveredDate|Price             |Region|OrderChannel|ReferredBy|InsertTimestamp     

In [21]:
channel_sales = spark.sql(f"""
SELECT
    OrderChannel,
    COUNT(OrderID) AS TotalOrders,
    SUM(CAST(Price AS DOUBLE)) AS TotalSales,
    AVG(CAST(Price AS DOUBLE)) AS AverageOrderValue
FROM {lakehouse_name}.{env_name}_silver_db.orders
GROUP BY OrderChannel
ORDER BY TotalSales DESC
""")

channel_sales.show()

channel_sales.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable(
    f"{lakehouse_name}.{env_name}_gold_db.gold_channel_sales"
)

print("Gold Channel Sales Created")

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 29, Finished, Available, Finished, False)

+------------+-----------+------------------+------------------+
|OrderChannel|TotalOrders|        TotalSales| AverageOrderValue|
+------------+-----------+------------------+------------------+
|  Mobile App|         11|           8004.74| 727.7036363636363|
|      Online|          8|3905.8700000000003|488.23375000000004|
|     Offline|          6|           2895.47| 482.5783333333333|
+------------+-----------+------------------+------------------+

Gold Channel Sales Created


In [22]:
kpi_df = spark.sql(f"""
SELECT
    COUNT(DISTINCT OrderID) AS TotalOrders,
    COUNT(DISTINCT CustomerID) AS TotalCustomers,
    SUM(CAST(Price AS DOUBLE)) AS TotalRevenue,
    AVG(CAST(Price AS DOUBLE)) AS AverageOrderValue
FROM {lakehouse_name}.{env_name}_silver_db.orders
""")

kpi_df.show()

kpi_df.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable(
    f"{lakehouse_name}.{env_name}_gold_db.gold_executive_kpi"
)

print("Executive KPI Created")

StatementMeta(, 1758c90a-59af-4ed7-aad0-befd82eeb062, 37, Finished, Available, Finished, False)

+-----------+--------------+------------------+-----------------+
|TotalOrders|TotalCustomers|      TotalRevenue|AverageOrderValue|
+-----------+--------------+------------------+-----------------+
|         25|            17|14806.080000000002|592.2432000000001|
+-----------+--------------+------------------+-----------------+

Executive KPI Created
